In [1]:
# Standard library
import math
import warnings
from pathlib import Path

# Third-party scientific stack
import numpy as np
import pandas as pd
from tqdm import tqdm

# Local project utilities
from utils.Processing import make_patient_dataframe, parse_mixcr, get_IBD_data

# Global settings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)

In [ ]:
# Define repository root directory
# All raw and processed data paths are constructed relative to this location
base_dir = Path("..").resolve()

# Discovery dataset
disc_data_dir = base_dir / "1_raw_data" / "0_discovery_data"
disc_raw_tcr_dir = disc_data_dir / "0_discovery_TCR"
disc_processed_dir = base_dir / "2_processed_data" / "0_discovery_data"
disc_processed_dir.mkdir(exist_ok=True)

# Brand et al. IBD dataset
ibd_data_dir = base_dir / "1_raw_data" / "1_Brand_data"
ibd_raw_alpha_dir = ibd_data_dir / "0_raw_data_TRA"
ibd_raw_beta_dir = ibd_data_dir / "0_raw_data_TRB"
ibd_processed_dir = base_dir / "2_processed_data" / "1_Brand_data"
ibd_processed_dir.mkdir(exist_ok=True)

# Pu et al. CRC dataset
crc_data_dir = base_dir / "1_raw_data" / "2_Pu_data"
crc_processed_dir = base_dir / "2_processed_data" / "2_Pu_data"
crc_processed_dir.mkdir(exist_ok=True)

# 1. Discovery dataset

## 1A. Process and parse raw TCR data

In [3]:
# Create directory for TCR files per patient
full_dir = disc_processed_dir / "1_patient_full"
full_dir.mkdir(exist_ok=True)

# Load the list with all patients included in this research
Patient_list = list(pd.read_csv(disc_data_dir / "Overlap_id_list.csv")["patient_id"])


# Parse raw MiXCR output per patient and cell type (CD4/CD8)
# Output: cleaned per-patient TCR clonotype tables
for patient in tqdm(Patient_list, desc="Processing patient dataframes"):
    patient_dfs = []

    for cell_type in ["CD4", "CD8"]:
        # Load raw data
        df = make_patient_dataframe(disc_raw_tcr_dir, patient, cell_type)
        if not df.empty:
            # Parse TCR data
            imgt_file = base_dir / "2_processed_data" / "imgt_reference.tsv"
            parsed_df = parse_mixcr(df, imgt_file=imgt_file, rename=True)
            parsed_df = parsed_df[
                [
                    "clone_id",
                    "duplicate_count",
                    "junction",
                    "junction_aa",
                    "v_call",
                    "j_call",
                    "refPoints",
                    "patient",
                    "cell_type",
                ]
            ]
            parsed_df = parsed_df.dropna().reset_index(drop=True)
            patient_dfs.append(parsed_df)

    if patient_dfs:
        # Combine CD4 and CD8 parsed data per patient
        df_full = pd.concat(patient_dfs, ignore_index=True)
        df_full.to_csv(full_dir / f"{patient}_parsed_full.csv", index=False)

print("All patient TCR data loaded, parsed and saved successfully.")


# Combine patient-level data and calculate clone counts/frequencies
full_df = pd.DataFrame()

for patient in tqdm(Patient_list, desc="Creating patient dataframes"):
    data = pd.read_csv(full_dir / f"{patient}_parsed_full.csv")

    data = data.drop(columns=["clone_id"])
    data["chain"] = data["v_call"].str.extract(r"(TR[ABDG])V.*")
    data["full_tcr"] = data["v_call"] + "_" + data["junction_aa"] + "_" + data["j_call"]

    # Make a clonecount column that calculates count separately for the separate cell types (CD4 - CD8)
    data["clonecount_celltype"] = data.groupby(["junction", "full_tcr", "cell_type"])[
        "duplicate_count"
    ].transform("sum")

    # Make a clonecount column that calculates count purely based on the full TCR and not the cell type
    data["clonecount"] = data.groupby(["junction", "full_tcr"])[
        "duplicate_count"
    ].transform("sum")

    # Calculate clone count frequencies either for cell types separately or for the full repertoire
    total_sum = data.drop_duplicates(subset=["junction", "full_tcr"])[
        "clonecount"
    ].sum()
    data["clonefreq_celltype"] = data["clonecount_celltype"] / total_sum
    data["clonefreq"] = data["clonecount"] / total_sum

    data = data.drop_duplicates(subset=["junction", "full_tcr", "cell_type"])

    full_df = pd.concat([full_df, data], ignore_index=True)

full_df.reset_index(drop=True, inplace=True)

print("All patient data combined into a single dataframe successfully.")

Processing patient dataframes: 100%|██████████| 74/74 [01:42<00:00,  1.38s/it]


All patient TCR data loaded, parsed and saved successfully.


Creating patient dataframes: 100%|██████████| 74/74 [01:06<00:00,  1.11it/s]

All patient data combined into a single dataframe successfully.


## 1B. Epitope annotation

### ImmuneWatch Detect formatting

In [4]:
imw_dir = disc_processed_dir / "IMW"
imw_dir.mkdir(exist_ok=True)

# Prepare IMW input files
IMW = full_df[["junction_aa", "v_call", "j_call"]].drop_duplicates()

# Split into chunks for IMW input
max_rows = 500_000
num_chunks = math.ceil(len(IMW) / max_rows)

for i in range(num_chunks):
    start = i * max_rows
    end = (i + 1) * max_rows
    IMW.iloc[start:end].to_csv(imw_dir / f"IMW_input_{i+1}.csv", index=False)

### Run IMW DETECT
In the terminal run this command:

(Adjust the range for the number of files you have and input the access key for DETECT)


for i in {1..9}; do      
    ./imw_detect \
        -i "path_to_processed_data/IMW/IMW_input_${i}.csv" \
        -o "path_to_processed_data/IMW/IMW_output_${i}.csv" \
        -d imwdb \
        -l {your access key}
done

### Add epitope annotations to the parsed dataframe

In [5]:
# Combine IMW output files
IMW_combined_list = []

for i in range(1, 10):
    imw_output = pd.read_csv(imw_dir / f"IMW_output_{i}.tsv", sep="\t")
    IMW_combined_list.append(imw_output)

IMW_combined = pd.concat(IMW_combined_list).drop_duplicates()

# Annotate main TCR dataframe with IMW epitope information
full_df = pd.merge(
    full_df,
    IMW_combined,
    on=["junction_aa", "v_call", "j_call"],
    how="left",
)

# Specific annotation for 5-OP-RU epitope
mask = full_df["Epitope"] == "5-OP-RU"
full_df.loc[mask, "Epitope Antigen"] = "Riboflavin (Vit. B)"
full_df.loc[mask, "Epitope Species"] = "Bacterial"

# Group epitope species in larger categories
species_categories = {
    "Viral": [
        "Human betaherpesvirus 5",
        "Influenza A virus",
        "Yellow fever virus",
        "Hepatitis B virus",
        "Synthetic,Hepatitis C virus",
        "Human gammaherpesvirus 4",
        "Influenza B virus,Influenza A virus",
        "Hepatitis C virus",
        "SARS-CoV,SARS-CoV-2",
        "SARS-CoV-2",
        "HPV",
        "HIV",
        "Human alphaherpesvirus 2",
        "Dengue Virus 1",
        "SARS-CoV-2,SARS-CoV",
        "HIV,HIV-1",
        "HIV-1,HIV",
        "HCoV-OC43,HCoV-HKU1",
        "Influenza B virus",
        "HIV-1",
        "Human alphaherpesvirus 3",
        "Human mastadenovirus C",
        "HTLV-1",
        "Human alphaherpesvirus 1",
        "Rotavirus",
        "HTLV-1,Homo sapiens",
        "Chrysanthemum virus B",
        "Human betaherpesvirus 5,Homo sapiens",
        "Homo sapiens,Influenza A virus",
        "Dengue Virus 3,Dengue Virus 4",
        "Adenoviridae",
    ],
    "Human": ["Homo sapiens", "Synthetic,Homo sapiens"],
    "Synthetic": ["Synthetic"],
    "Microbial": [
        "Bacterial",
        "Streptococcus pneumoniae",
        "Triticum aestivum",
        "Mycobacterium tuberculosis",
        "Plasmodium falciparum",
        "Pseudomonas aeruginosa",
        "Azospirillum",
        "Escherichia coli K-12",
    ],
}


def categorize_species(species):
    for category, species_list in species_categories.items():
        if species in species_list:
            return category
    return np.nan


# Apply species categorization
full_df["epitope_groups"] = full_df["Epitope Species"].apply(categorize_species)

full_df.to_csv(disc_processed_dir / "0_final_data_parsed_annotated.csv", index=False)

print(f"Final annotated TCR dataframe saved")

Final annotated TCR dataframe saved


# 2. Brand et al. independent dataset

In [6]:
# Load the raw datafiles
Alpha = get_IBD_data(ibd_raw_alpha_dir, ibd_data_dir / "1_metadata_TRA.xlsx")
Alpha["chain"] = "TRA"

Beta = get_IBD_data(ibd_raw_beta_dir, ibd_data_dir / "1_metadata_TRB.xlsx")
Beta["chain"] = "TRB"

# Combine alpha and beta into one dataframe
Full_df = pd.concat([Alpha, Beta])

# Parse the TCR data
imgt_file = base_dir / "2_processed_data" / "imgt_reference.tsv"
Full_parsed = parse_mixcr(Full_df, imgt_file=imgt_file, rename=True)
Full_parsed = Full_parsed[
    [
        "clone_id",
        "duplicate_count",
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "chain",
        "refPoints",
        "Sample",
        "patient_id",
        "cell_type",
        "Sex",
        "Age",
        "zyg_def",
        "Concordancy",
        "Diagnosis",
        "Inflammation_rectoscopy",
        "Total_PBMCs_isolated",
    ]
]

Full_parsed = Full_parsed.dropna(
    subset={
        "clone_id",
        "duplicate_count",
        "junction",
        "junction_aa",
        "v_call",
        "j_call",
        "refPoints",
    }
).reset_index(drop=True)

# Extract the TCR chain and add the full VDJ information
Full_parsed["chain_corrected"] = Full_parsed["v_call"].str.extract(r"(TR[A,B])")

Full_parsed["full_tcr"] = (
    Full_parsed["v_call"]
    + "_"
    + Full_parsed["junction_aa"]
    + "_"
    + Full_parsed["j_call"]
)

# Add clonecount per full tcr and cell type
Full_parsed["clonecount_celltype"] = Full_parsed.groupby(
    ["full_tcr", "patient_id", "cell_type"]
)["duplicate_count"].transform("sum")

# Add clonedount per full tcr
Full_parsed["clonecount"] = Full_parsed.groupby(["full_tcr", "patient_id"])[
    "duplicate_count"
].transform("sum")

# For each patient get the total clone count within that patient
Full_parsed["Total_sum"] = Full_parsed.groupby(["patient_id"])[
    "duplicate_count"
].transform("sum")

# Get clone frequencies for full tcr or full tcr combined with cell type
Full_parsed["clonefreq_celltype"] = (
    Full_parsed["clonecount_celltype"] / Full_parsed["Total_sum"]
)

Full_parsed["clonefreq"] = Full_parsed["clonecount"] / Full_parsed["Total_sum"]

# Drop duplicate entries
Full_parsed = Full_parsed.drop_duplicates(
    subset=["full_tcr", "patient_id", "cell_type"]
)

Full_parsed.to_csv(ibd_processed_dir / "0_parsed_full_IBD_data.csv", index=False)

print(f"Final IBD parsed TCR dataframe saved")

Final IBD parsed TCR dataframe saved


# 3. Pu et al. independent dataset

In [7]:
# Read in the TCR data and drop duplicates
crc_data = pd.read_csv(crc_data_dir / "0_Janssen_TCR.csv")
crc_data = crc_data.drop_duplicates(
    subset=["sample_tissue", "clone_id_aa_identity", "cell_type_groups"]
)

# Save to processed data
crc_data.to_csv(crc_processed_dir / "0_Janssen_TCR.csv")

### Microbiome parsing: Transform the ASV counts into an abundance matrix per sample

In [8]:
# Read in the ASV counts and their associated taxonomy
micro_counts = pd.read_csv(crc_data_dir / "1_ASVs_counts.tsv", sep="\t", index_col=0)
taxa_names = pd.read_csv(crc_data_dir / "2_ASVs_taxonomy.tsv", sep="\t").rename(
    columns={"Unnamed: 0": "ASV", "Species": "Species_full"}
)

# Add patient names per sample ID
patient_names = pd.read_excel(crc_data_dir / "3_microbiome_fastqs_to_patient_ids.xlsx")


# Identify the most detailed taxon known per ASV
taxa_names["Species"] = taxa_names["Species_full"].str.split("(").str[0]
taxonomy_levels = ["Genus", "Family", "Order", "Class", "Phylum", "Kingdom"]
taxa_names["most_detailed_taxon"] = taxa_names[taxonomy_levels].bfill(axis=1).iloc[:, 0]


# Generate a column with taxon name containing this most detailed taxon
def generate_taxon_name(taxon):
    if taxon not in taxon_counts:
        taxon_counts[taxon] = 1
    else:
        taxon_counts[taxon] += 1
    return f"{taxon} {taxon_counts[taxon]}"


taxon_counts = {}
taxa_names["taxon_name"] = taxa_names["most_detailed_taxon"].apply(generate_taxon_name)
taxa_dict = dict(zip(taxa_names["ASV"], taxa_names["taxon_name"]))
micro_counts = micro_counts.rename(index=taxa_dict)

# Adjust patient identifiers
patient_names["type"] = patient_names["type"].replace(
    {"tumor": "Tumor", "normal": "Normal"}
)
patient_names["patient"] = patient_names["ID"] + "_" + patient_names["type"]
patient_dict = dict(zip(patient_names["fqID"], patient_names["patient"]))
micro_counts = micro_counts.rename(columns=patient_dict)

# Transpose the microbiome count matrix so that the samples are the rows
micro_counts = micro_counts.T
micro_counts = micro_counts.div(micro_counts.sum(axis=1), axis=0)

# Save the matrix to the processed data
micro_counts.to_csv(crc_processed_dir / "1_Micro_count_matrix_full.csv")